# optimizer-class-dispatch — ex1: build an optimizer from a config string via a class-dispatch dict

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-class-dispatch`. Running the final beacon cell reports progress against the `Config: Optimizer class dispatch` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: Optimizer class dispatch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-class-dispatch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-class-dispatch"
DD_SUBTOPIC = "Config: Optimizer class dispatch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Config: Optimizer class dispatch — quick refresher

Switching optimizers based on a config string is universal in training scripts. The idiom:

```python
OPTIMIZER_CLASSES = {
    'sgd':  torch.optim.SGD,
    'adam': torch.optim.Adam,
    'adamw': torch.optim.AdamW,
}
optimizer = OPTIMIZER_CLASSES[cfg.optimizer_name](
    model.parameters(), lr=cfg.lr,
)
```

**Why a dict, not `if/elif/else`.** The dict is data — you can register a new optimizer from a plugin, iterate the keys for a CLI `--help`, or test the dispatch table without instantiating any optimizer. An `if/elif` chain hides this in control flow.

**Store CLASSES, not instances.** The dict holds the class itself (`SGD`, not `SGD(...)`), because the instance needs `model.parameters()` and per-run kwargs. The class is the factory; you call it at construction time.

**Why a `KeyError` is the right failure mode.** If the user passes `--optimizer rmsprop` and you forgot to register it, `KeyError('rmsprop')` halts the run cleanly with the offending name in the message. Wrapping it in a try/except that returns `None` produces a much more confusing downstream failure.

### Exercise 1 — build an optimizer from a config string via a class-dispatch dict

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the dispatch-dict factory pattern `OPTIMIZER_CLASSES[name](params, lr=...)` to construct the correct `torch.optim` optimizer from a string.
> Keywords: dispatch-table, registry, factory, optimizer-name
> ```

**KCs targeted:** `class-dispatch-dict-lookup`, `optimizer-factory-call`

Implement `ex1_build_optimizer(name, params, lr)`. The config-string → optimizer-instance dispatch.

1. Define a module-level dict `OPTIMIZER_CLASSES` mapping `'sgd'`, `'adam'`, `'adamw'` to `torch.optim.SGD`, `torch.optim.Adam`, `torch.optim.AdamW` respectively.
2. `ex1_build_optimizer` looks up the class by `name`, calls it with `(params, lr=lr)`, and returns the instance.
3. An unknown `name` should raise `KeyError` with the offending name in the message. The default dict indexing already does this.

Inputs:
- `name`: `str` from `{'sgd', 'adam', 'adamw'}`.
- `params`: iterable of parameter tensors.
- `lr`: float learning rate.

Output: `torch.optim.Optimizer` instance.

Why this matters: dispatching by string is how real training scripts let you sweep optimizer choice from a config file without code changes.

In [ ]:
OPTIMIZER_CLASSES = {
    # Fill in: 'sgd', 'adam', 'adamw' -> torch.optim classes.
}

def ex1_build_optimizer(name: str, params, lr: float):
    """Look up name in OPTIMIZER_CLASSES and instantiate."""
    raise NotImplementedError()


def _test_ex1():
    # === Module-level dispatch dict ===
    assert isinstance(OPTIMIZER_CLASSES, dict), 'OPTIMIZER_CLASSES must be a dict'
    assert set(OPTIMIZER_CLASSES.keys()) == {'sgd', 'adam', 'adamw'}, (
        f'OPTIMIZER_CLASSES keys must be {{sgd, adam, adamw}}, got {set(OPTIMIZER_CLASSES.keys())}'
    )
    # Dict stores CLASSES, not instances.
    assert OPTIMIZER_CLASSES['sgd'] is t.optim.SGD
    assert OPTIMIZER_CLASSES['adam'] is t.optim.Adam
    assert OPTIMIZER_CLASSES['adamw'] is t.optim.AdamW
    # These are types, not optimizer objects.
    for cls in OPTIMIZER_CLASSES.values():
        assert isinstance(cls, type), f'expected a class, got {cls!r}'

    # === Construct each kind ===
    param = t.nn.Parameter(t.randn(3))
    for name, expected_cls in [
        ('sgd', t.optim.SGD),
        ('adam', t.optim.Adam),
        ('adamw', t.optim.AdamW),
    ]:
        opt = ex1_build_optimizer(name, [param], lr=1e-2)
        assert isinstance(opt, expected_cls), (
            f'name={name!r}: expected {expected_cls.__name__}, got {type(opt).__name__}'
        )
        assert opt.param_groups[0]['lr'] == 1e-2, f'lr not propagated for {name}'
        assert len(opt.param_groups[0]['params']) == 1

    # === Round-trip a tiny step to prove the optimizer actually works ===
    p2 = t.nn.Parameter(t.zeros(2))
    opt2 = ex1_build_optimizer('sgd', [p2], lr=0.1)
    p2.grad = t.tensor([1.0, -2.0])
    opt2.step()
    # SGD: p_new = p - lr * grad = -0.1 * [1, -2] = [-0.1, 0.2]
    assert t.allclose(p2.detach(), t.tensor([-0.1, 0.2]), atol=1e-6), (
        f'SGD step produced wrong value: {p2.detach()}'
    )

    # === Unknown name => KeyError ===
    try:
        ex1_build_optimizer('rmsprop', [t.nn.Parameter(t.randn(2))], lr=1e-3)
    except KeyError as e:
        # The KeyError's args should mention the unknown name.
        assert 'rmsprop' in str(e), f'KeyError should mention the bad name, got {e!r}'
    else:
        raise AssertionError('expected KeyError for unknown optimizer name')

    # === Different LRs construct independent optimizers ===
    opts = [
        ex1_build_optimizer('adam', [t.nn.Parameter(t.randn(3))], lr=lr)
        for lr in [1e-4, 3e-4, 1e-3]
    ]
    lrs = [o.param_groups[0]['lr'] for o in opts]
    assert lrs == [1e-4, 3e-4, 1e-3], f'per-call lr not isolated, got {lrs}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
OPTIMIZER_CLASSES = {
    'sgd':   t.optim.SGD,
    'adam':  t.optim.Adam,
    'adamw': t.optim.AdamW,
}

def ex1_build_optimizer(name, params, lr):
    cls = OPTIMIZER_CLASSES[name]
    return cls(params, lr=lr)
```

**Why `OPTIMIZER_CLASSES[name]` not `getattr(t.optim, name)`.** The `getattr` form would let users pass any attribute of the `torch.optim` module — including `Optimizer` (the abstract base), `lr_scheduler`, etc. An explicit registry is a SECURITY/CORRECTNESS boundary: only the names you registered are reachable.

**Letting `KeyError` bubble.** A common anti-pattern is `try: ... except KeyError: return None`. Now the caller downstream gets `AttributeError: 'NoneType' object has no attribute 'step'` 10 stack frames deep with no hint about the config typo. Don't swallow it.

**Extending the registry.** Want to add Lion or Sophia? Just `OPTIMIZER_CLASSES['lion'] = LionOptimizer` before the dispatch runs. The factory function itself doesn't change — that's the whole win over `if/elif`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()